# What `dataset_full.csv` contains

This notebook describes the dataset, checks its quality, and explains how the RehaPiano measurements are structured.

The analysis answers five questions:

1. What is the unit of observation, and how many participants, sessions, and exercises are present?
2. Which fields does the table contain, and how are they related?
3. Are there missing values, duplicates, or structural inconsistencies?
4. What is stored in the nested JSON in `additional_data`?
5. Which limitations should be considered before statistical analysis or modeling?

The notebook does not modify the source CSV. All derived tables are created in memory only.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("../../data/RehaPiano/datasets/dataset_full.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "dataset_full.csv not found; see data/README.md for the data layout, "
        "or update DATA_PATH."
    )

raw = pd.read_csv(DATA_PATH, sep=";")
df = raw.copy()
df["timestamp_parsed"] = pd.to_datetime(df["timestamp"], format="mixed", errors="coerce")
df["additional_data_parsed"] = df["additional_data"].map(json.loads)

display(Markdown(
    f"**File loaded:** {len(df):,} rows × {raw.shape[1]} source columns. "
    f"The delimiter is a semicolon (`;`)."
))

**File loaded:** 910 rows × 12 source columns. The delimiter is a semicolon (`;`).

## 1. Unit of observation

Each row represents **one step/exercise within a measurement session**, not one patient and not an individual key press.

- `patient_id` links rows belonging to the same participant.
- `measurement_id` identifies a measurement session.
- `step_id` and `step_type` identify the performed exercise.
- `filename` points to a related PKL file containing raw signals; the PKL files themselves are not included in the CSV.
- Reaction and rhythm events are nested inside the JSON string in `additional_data`.

In [12]:
overview = pd.Series({
    "Rows": len(df),
    "Source columns": raw.shape[1],
    "Unique participants": df["patient_id"].nunique(),
    "Measurement sessions": df["measurement_id"].nunique(),
    "Unique steps (step_id)": df["step_id"].nunique(),
    "Exercise types": df["step_type"].nunique(),
    "Unique related PKL files": df["filename"].nunique(),
    "Period start": df["timestamp_parsed"].min(),
    "Period end": df["timestamp_parsed"].max(),
}, name="Value").to_frame()
display(overview)

display(Markdown("### First rows (without the long JSON field)"))
display(df.drop(columns=["additional_data", "additional_data_parsed"]).head())

,Value
Rows,910
Source columns,12
Unique participants,138
Measurement sessions,173
Unique steps (step_id),9
Exercise types,4
Unique related PKL files,775
Period start,2025-02-13 08:12:06
Period end,2026-09-09 15:41:10


### First rows (without the long JSON field)

,patient_id,timestamp,age,height,weight,gender,hand,measurement_id,step_id,step_type,filename,timestamp_parsed
0,1302202501,2/13/2025 8:12:06,23,157,80,female,right,e495759d-9eb7-44c9-9216-c67d24187d87,5867ebbc-a463-4d20-8b20-50df7c64ace7,EXACT_FORCE,e495759d-9eb7-44c9-9216-c67d24187d87_5867ebbc-a463-4d20-8b20-50df7c64ace7.pkl,2025-02-13 08:12:06
1,1302202501,2/13/2025 8:12:06,23,157,80,female,right,e495759d-9eb7-44c9-9216-c67d24187d87,35b5ee83-1630-4f8d-af98-59fde4bf0e98,REACTION,e495759d-9eb7-44c9-9216-c67d24187d87_35b5ee83-1630-4f8d-af98-59fde4bf0e98.pkl,2025-02-13 08:12:06
2,1302202501,2/13/2025 8:12:06,23,157,80,female,right,e495759d-9eb7-44c9-9216-c67d24187d87,1e177cdd-0baf-42fe-b297-fbeba73ea828,RHYTHM,e495759d-9eb7-44c9-9216-c67d24187d87_1e177cdd-0baf-42fe-b297-fbeba73ea828.pkl,2025-02-13 08:12:06
3,1302202501,2/13/2025 8:12:06,23,157,80,female,right,e495759d-9eb7-44c9-9216-c67d24187d87,d475bc4a-2c13-4297-98f9-cab77413ac2b,EXACT_FORCE,e495759d-9eb7-44c9-9216-c67d24187d87_d475bc4a-2c13-4297-98f9-cab77413ac2b.pkl,2025-02-13 08:12:06
4,1302202502,2/13/2025 8:15:37,21,185,65,male,right,78916fbd-f255-4db5-b677-33b13e555f0a,5867ebbc-a463-4d20-8b20-50df7c64ace7,EXACT_FORCE,78916fbd-f255-4db5-b677-33b13e555f0a_5867ebbc-a463-4d20-8b20-50df7c64ace7.pkl,2025-02-13 08:15:37


## 2. Data dictionary

The descriptions below are based on the CSV structure, observed values, and conventions in the source project. `height` and `weight` appear to use centimeters and kilograms. Offset and reaction values appear to use milliseconds, but the protocol specification should be consulted to confirm this.

In [3]:
dictionary = pd.DataFrame([
    ("patient_id", "Participant identifier", "string", "Numeric and coded identifiers in two formats"),
    ("timestamp", "Session date and time", "string → datetime", "The file contains mixed date formats"),
    ("age", "Age", "integer", "Years"),
    ("height", "Height", "integer", "Presumably centimeters"),
    ("weight", "Weight", "integer", "Presumably kilograms"),
    ("gender", "Gender", "category", "female / male"),
    ("hand", "Dominant hand", "category", "left / right; some values are missing"),
    ("measurement_id", "Measurement-session identifier", "UUID", "Links exercises from the same session"),
    ("step_id", "Exercise-scenario identifier", "UUID", "9 values; associated with protocol type"),
    ("step_type", "Exercise type", "category", "EXACT_FORCE, MAX_FORCE, REACTION, RHYTHM"),
    ("additional_data", "Exercise metadata", "JSON string", "Events for REACTION/RHYTHM; empty object for force steps"),
    ("filename", "Related raw-data filename", "string", "measurement_id + step_id + .pkl"),
], columns=["Field", "Contents", "Type", "Comment"])
display(dictionary)

,Field,Contents,Type,Comment
0,patient_id,Participant identifier,string,Numeric and coded identifiers in two formats
1,timestamp,Session date and time,string → datetime,The file contains mixed date formats
2,age,Age,integer,Years
3,height,Height,integer,Presumably centimeters
4,weight,Weight,integer,Presumably kilograms
5,gender,Gender,category,female / male
6,hand,Dominant hand,category,left / right; some values are missing
7,measurement_id,Measurement-session identifier,UUID,Links exercises from the same session
8,step_id,Exercise-scenario identifier,UUID,9 values; associated with protocol type
9,step_type,Exercise type,category,"EXACT_FORCE, MAX_FORCE, REACTION, RHYTHM"


## 3. Data quality

In [4]:
expected_filename = (
    df["measurement_id"].astype(str) + "_" + df["step_id"].astype(str) + ".pkl"
)
json_ok = []
for value in df["additional_data"]:
    try:
        json.loads(value)
        json_ok.append(True)
    except (TypeError, json.JSONDecodeError):
        json_ok.append(False)

quality = pd.DataFrame({
    "Check": [
        "Exactly duplicated rows",
        "Repeated measurement_id + step_id pair",
        "Non-unique filename",
        "Unparseable timestamp",
        "Invalid additional_data JSON",
        "filename does not match measurement_id + step_id",
    ],
    "Count": [
        int(df.duplicated(subset=raw.columns.tolist()).sum()),
        int(df.duplicated(["measurement_id", "step_id"]).sum()),
        int(df["filename"].duplicated().sum()),
        int(df["timestamp_parsed"].isna().sum()),
        int((~pd.Series(json_ok)).sum()),
        int((df["filename"] != expected_filename).sum()),
    ],
})
display(quality)

missing = pd.DataFrame({
    "Missing": raw.isna().sum(),
    "% of rows": raw.isna().mean().mul(100).round(1),
    "Unique values": raw.nunique(dropna=True),
    "pandas dtype": raw.dtypes.astype(str),
}).sort_values(["Missing", "Unique values"], ascending=[False, True])
display(Markdown("### Missing values, cardinality, and types"))
display(missing)

display(Markdown(
    "**Conclusion.** The JSON and filenames are structurally consistent. The main quality issues are "
    "135 exact duplicates and missing `hand` values. Timestamps parse correctly only "
    "when mixed formats are supported."
))

,Check,Count
0,Exactly duplicated rows,135
1,Repeated measurement_id + step_id pair,135
2,Non-unique filename,135
3,Unparseable timestamp,0
4,Invalid additional_data JSON,0
5,filename does not match measurement_id + step_id,0


### Missing values, cardinality, and types

,Missing,% of rows,Unique values,pandas dtype
hand,550,60.40,2,object
gender,0,0.00,2,object
step_type,0,0.00,4,object
step_id,0,0.00,9,object
height,0,0.00,37,int64
age,0,0.00,53,int64
weight,0,0.00,53,int64
patient_id,0,0.00,138,object
timestamp,0,0.00,173,object
measurement_id,0,0.00,173,object


**Conclusion.** The JSON and filenames are structurally consistent. The main quality issues are 135 exact duplicates and missing `hand` values. Timestamps parse correctly only when mixed formats are supported.

In [5]:
# Remove exact duplicates only for the subsequent content analysis.
duplicate_mask = df.duplicated(subset=raw.columns.tolist())
clean = df.loc[~duplicate_mask].copy()

duplicates_by_session = (
    df[df.duplicated(subset=raw.columns.tolist(), keep=False)]
    .groupby("measurement_id")
    .size()
    .value_counts()
    .rename_axis("Rows in an affected session")
    .to_frame("Number of sessions")
)

display(Markdown(
    f"After removing exact duplicates, **{len(clean):,} rows** remain "
    f"({len(df) - len(clean):,} removed). Duplicates are concentrated in specific sessions:"
))
display(duplicates_by_session)

After removing exact duplicates, **775 rows** remain (135 removed). Duplicates are concentrated in specific sessions:

,Number of sessions
Rows in an affected session,
10,27


## 4. Participants and demographics

To avoid inflating the sample with repeated sessions, demographics are calculated from one record per `patient_id`.

In [6]:
patients = (
    clean.sort_values("timestamp_parsed")
    .drop_duplicates("patient_id")
    .copy()
)
patients["bmi"] = patients["weight"] / (patients["height"] / 100) ** 2

numeric_demo = patients[["age", "height", "weight", "bmi"]].describe().T
numeric_demo = numeric_demo.rename(index={
    "age": "Age, years",
    "height": "Height, cm (assumed)",
    "weight": "Weight, kg (assumed)",
    "bmi": "BMI (derived)",
})
display(numeric_demo)

for column, title in [("gender", "Gender"), ("hand", "Dominant hand")]:
    table = patients[column].value_counts(dropna=False).rename_axis(title).to_frame("Participants")
    table["%"] = (table["Participants"] / len(patients) * 100).round(1)
    display(table)

id_type = pd.Series({
    "Digits only": patients["patient_id"].astype(str).str.fullmatch(r"\d+").sum(),
    "Hyphenated code": patients["patient_id"].astype(str).str.fullmatch(r"[A-Z0-9]+-[A-Z0-9]+").sum(),
}, name="Participants").to_frame()
display(Markdown("### patient_id formats"))
display(id_type)

sessions_per_patient = clean.groupby("patient_id")["measurement_id"].nunique()
display(Markdown("### Sessions per participant"))
display(sessions_per_patient.value_counts().sort_index().rename_axis("Sessions").to_frame("Participants"))

,count,mean,std,min,25%,50%,75%,max
"Age, years",138.00,45.01,22.80,18.00,22.25,43.00,65.75,89.00
"Height, cm (assumed)",138.00,173.60,10.00,153.00,165.00,174.00,181.00,205.00
"Weight, kg (assumed)",138.00,79.04,17.55,41.00,66.25,77.50,90.00,150.00
BMI (derived),138.00,26.25,5.50,16.56,22.02,25.57,30.98,42.24


,Participants,%
Gender,,
male,82,59.40
female,56,40.60


,Participants,%
Dominant hand,,
right,77,55.80
NaN,48,34.80
left,13,9.40


### patient_id formats

,Participants
Digits only,90
Hyphenated code,48


### Sessions per participant

,Participants
Sessions,
1,103
2,35


## 5. Measurement-session composition

After exact duplicates are removed, sessions follow two stable protocol variants containing either four or five rows. `REACTION` and `RHYTHM` occur in every session. The five-row variant adds two `MAX_FORCE` steps, while the four-row variant contains two `EXACT_FORCE` steps.

In [7]:
step_counts = clean["step_type"].value_counts().to_frame("Rows after duplicate removal")
step_counts["%"] = (step_counts.iloc[:, 0] / len(clean) * 100).round(1)
display(step_counts)

session_composition = (
    clean.groupby("measurement_id")["step_type"]
    .apply(lambda s: " + ".join(sorted(s)))
    .value_counts()
    .rename_axis("Session composition")
    .to_frame("Sessions")
)
display(session_composition)

step_map = pd.crosstab(clean["step_type"], clean["step_id"])
display(Markdown("### Relationship between step_type and step_id"))
display(step_map)

,Rows after duplicate removal,%
step_type,,
EXACT_FORCE,263,33.90
REACTION,173,22.30
RHYTHM,173,22.30
MAX_FORCE,166,21.40


,Sessions
Session composition,
EXACT_FORCE + EXACT_FORCE + REACTION + RHYTHM,90
EXACT_FORCE + MAX_FORCE + MAX_FORCE + REACTION + RHYTHM,83


### Relationship between step_type and step_id

step_id,1e177cdd-0baf-42fe-b297-fbeba73ea828,35b5ee83-1630-4f8d-af98-59fde4bf0e98,38950659-e401-4c26-8fd2-f7a6aca76595,5867ebbc-a463-4d20-8b20-50df7c64ace7,65a5a4b1-907c-4717-b47b-cd8f31eb3e4b,7d6d0186-fc1d-4280-996b-72b9db8490f3,81e85188-e29c-44cf-bf64-d2777af9c1e3,d2e6b7bd-a952-4652-afe3-9888e29ac205,d475bc4a-2c13-4297-98f9-cab77413ac2b
step_type,,,,,,,,,
EXACT_FORCE,0,0,0,90,0,0,0,83,90
MAX_FORCE,0,0,0,0,0,83,83,0,0
REACTION,0,90,83,0,0,0,0,0,0
RHYTHM,90,0,0,0,83,0,0,0,0


## 6. What `additional_data` contains

This field stores serialized JSON in two fundamentally different forms:

- for `EXACT_FORCE` and `MAX_FORCE`, it is an empty object `{}`; the numeric force signal is probably stored only in the related PKL file;
- for `REACTION`, it is a list of four event series with the fields `finger`, `offset`, and `reaction`;
- for `RHYTHM`, it is a list of four event series with the fields `press_offset` and `release_offset`.

In [8]:
def json_shape(value):
    if isinstance(value, dict):
        return f"object: keys={sorted(value)}"
    if isinstance(value, list):
        nested = all(isinstance(x, list) for x in value)
        return f"list: outer={len(value)}, nested_lists={nested}"
    return type(value).__name__

json_summary = (
    clean.assign(json_shape=clean["additional_data_parsed"].map(json_shape))
    .groupby(["step_type", "json_shape"], dropna=False)
    .size()
    .to_frame("Rows")
)
display(json_summary)

examples = []
for step_type, group in clean.groupby("step_type", sort=True):
    obj = group["additional_data_parsed"].iloc[0]
    if isinstance(obj, list):
        sample = obj[0][:2] if obj and isinstance(obj[0], list) else obj[:2]
    else:
        sample = obj
    examples.append((step_type, json.dumps(sample, ensure_ascii=False)))
display(pd.DataFrame(examples, columns=["step_type", "Abbreviated JSON example"]))

,,Rows
step_type,json_shape,
EXACT_FORCE,object: keys=[],263
MAX_FORCE,object: keys=[],166
REACTION,"list: outer=4, nested_lists=True",173
RHYTHM,"list: outer=4, nested_lists=True",173


,step_type,Abbreviated JSON example
0,EXACT_FORCE,{}
1,MAX_FORCE,{}
2,REACTION,"[{""finger"": 0, ""offset"": 1135, ""reaction"": 0}, {""finger"": 8, ""offset"": 4259, ""reaction..."
3,RHYTHM,"[{""press_offset"": 1382, ""release_offset"": 1784}, {""press_offset"": 2619, ""release_offse..."


## 7. Reaction events (`REACTION`)

In [9]:
reaction_rows = []
for _, row in clean.loc[clean["step_type"].eq("REACTION")].iterrows():
    for trial_no, trial in enumerate(row["additional_data_parsed"], start=1):
        for event in trial:
            reaction_rows.append({
                "patient_id": row["patient_id"],
                "measurement_id": row["measurement_id"],
                "trial": trial_no,
                **event,
            })

reaction = pd.DataFrame(reaction_rows)
nonzero_reaction = reaction.loc[reaction["reaction"].gt(0), "reaction"]

reaction_summary = pd.Series({
    "REACTION rows (sessions)": clean["step_type"].eq("REACTION").sum(),
    "Total events": len(reaction),
    "Mean events per session": reaction.groupby("measurement_id").size().mean(),
    "Minimum events per session": reaction.groupby("measurement_id").size().min(),
    "Maximum events per session": reaction.groupby("measurement_id").size().max(),
    "Events with reaction = 0": reaction["reaction"].eq(0).sum(),
    "Share with reaction = 0, %": reaction["reaction"].eq(0).mean() * 100,
    "Median non-zero reaction": nonzero_reaction.median(),
    "Mean non-zero reaction": nonzero_reaction.mean(),
    "Minimum non-zero reaction": nonzero_reaction.min(),
    "Maximum non-zero reaction": nonzero_reaction.max(),
}, name="Value").to_frame()
display(reaction_summary)

by_finger = reaction.groupby("finger").agg(
    events=("reaction", "size"),
    zero_share=("reaction", lambda s: s.eq(0).mean()),
    median_nonzero_reaction=("reaction", lambda s: s[s.gt(0)].median()),
)
by_finger["zero_share"] = (by_finger["zero_share"] * 100).round(1)
display(Markdown("### Summary by finger code (0–9)"))
display(by_finger)

display(Markdown(
    "`reaction = 0` occurs frequently. It may indicate a missed stimulus, no valid "
    "response, or a special protocol code. A zero should not be interpreted as an actual "
    "reaction time without documentation. Non-zero values are probably measured in milliseconds."
))

,Value
REACTION rows (sessions),173.00
Total events,"7,579.00"
Mean events per session,43.81
Minimum events per session,36.00
Maximum events per session,61.00
Events with reaction = 0,"3,295.00"
"Share with reaction = 0, %",43.48
Median non-zero reaction,"1,001.00"
Mean non-zero reaction,"1,057.42"
Minimum non-zero reaction,1.00


### Summary by finger code (0–9)

,events,zero_share,median_nonzero_reaction
finger,,,
0,742,66.60,969.00
1,758,54.10,"1,075.50"
2,759,54.90,"1,047.00"
3,773,30.70,986.00
4,765,12.70,968.50
5,768,15.90,970.00
6,755,29.90,981.00
7,740,55.80,"1,051.00"
8,758,56.10,"1,081.00"


`reaction = 0` occurs frequently. It may indicate a missed stimulus, no valid response, or a special protocol code. A zero should not be interpreted as an actual reaction time without documentation. Non-zero values are probably measured in milliseconds.

## 8. Rhythm events (`RHYTHM`)

In [10]:
rhythm_rows = []
for _, row in clean.loc[clean["step_type"].eq("RHYTHM")].iterrows():
    for trial_no, trial in enumerate(row["additional_data_parsed"], start=1):
        for event_no, event in enumerate(trial, start=1):
            rhythm_rows.append({
                "patient_id": row["patient_id"],
                "measurement_id": row["measurement_id"],
                "trial": trial_no,
                "event_no": event_no,
                **event,
            })

rhythm = pd.DataFrame(rhythm_rows)
rhythm["duration"] = rhythm["release_offset"] - rhythm["press_offset"]
valid_duration = rhythm.loc[rhythm["release_offset"].gt(0), "duration"]

rhythm_summary = pd.Series({
    "RHYTHM rows (sessions)": clean["step_type"].eq("RHYTHM").sum(),
    "Total press events": len(rhythm),
    "Mean events per session": rhythm.groupby("measurement_id").size().mean(),
    "Minimum events per session": rhythm.groupby("measurement_id").size().min(),
    "Maximum events per session": rhythm.groupby("measurement_id").size().max(),
    "Events with release_offset = 0": rhythm["release_offset"].eq(0).sum(),
    "Share with release_offset = 0, %": rhythm["release_offset"].eq(0).mean() * 100,
    "Median press duration": valid_duration.median(),
    "Mean press duration": valid_duration.mean(),
    "Minimum press duration": valid_duration.min(),
    "Maximum press duration": valid_duration.max(),
}, name="Value").to_frame()
display(rhythm_summary)

events_per_trial = rhythm.groupby(["measurement_id", "trial"]).size()
display(Markdown("### Events per trial"))
display(events_per_trial.describe().to_frame("Events"))

display(Markdown(
    "`release_offset = 0` should be treated as an unfinished or missing release, "
    "so these rows are excluded from the duration calculation. Offsets and durations "
    "appear to use milliseconds, but the unit should be confirmed."
))

,Value
RHYTHM rows (sessions),173.00
Total press events,"44,492.00"
Mean events per session,257.18
Minimum events per session,1.00
Maximum events per session,566.00
Events with release_offset = 0,257.00
"Share with release_offset = 0, %",0.58
Median press duration,144.00
Mean press duration,171.68
Minimum press duration,14.00


### Events per trial

,Events
count,686.00
mean,64.86
std,34.70
min,1.00
25%,38.00
50%,62.00
75%,90.00
max,158.00


`release_offset = 0` should be treated as an unfinished or missing release, so these rows are excluded from the duration calculation. Offsets and durations appear to use milliseconds, but the unit should be confirmed.

## 9. Overall interpretation

`dataset_full.csv` is an **index and metadata table for RehaPiano measurement sessions**, supplemented with participant questionnaire attributes and event data for two exercises.

Key findings:

1. The source file contains 910 rows, 138 participants, and 173 sessions. Removing 135 exact duplicates leaves 775 unique step rows.
2. There are two session-protocol variants: 90 sessions with four steps and 83 sessions with five steps.
3. Demographic attributes are constant within each `patient_id`. Participant-level analysis should aggregate by participant, while repeated-measures analysis must account for sessions belonging to the same participant.
4. `hand` is missing for part of the cohort. The missingness is associated with the data-collection structure and does not appear random.
5. `REACTION` and `RHYTHM` contain event annotations. Force values for `EXACT_FORCE` and `MAX_FORCE` are absent from the CSV and are probably stored in the PKL files referenced by `filename`.
6. Zero values in `reaction` and `release_offset` represent control codes or incomplete events. Their meaning should be confirmed before clinical metrics are calculated.
7. The formats of `patient_id`, timestamps, and session composition suggest that multiple batches or cohorts were combined. For modeling, train/test splitting should be performed by `patient_id`, not by row, to prevent data leakage.

### Items to clarify before further analysis

- the exact units of `offset`, `reaction`, height, and weight;
- how `finger` codes 0–9 map to fingers and hands;
- the meaning of zero `reaction` and `release_offset` values;
- the content and decoding method of the related PKL files;
- whether the two protocol variants represent separate studies or versions of the same protocol.


In [11]:
# Compact machine-checkable summary for reproducible execution.
final_check = pd.Series({
    "Source rows": len(df),
    "Rows after duplicate removal": len(clean),
    "Participants": clean["patient_id"].nunique(),
    "Sessions": clean["measurement_id"].nunique(),
    "REACTION events": len(reaction),
    "RHYTHM events": len(rhythm),
    "Timestamp parsing errors": clean["timestamp_parsed"].isna().sum(),
    "JSON parsing errors": 0,
}, name="Value").to_frame()
display(final_check)

,Value
Source rows,910
Rows after duplicate removal,775
Participants,138
Sessions,173
REACTION events,7579
RHYTHM events,44492
Timestamp parsing errors,0
JSON parsing errors,0
